# Bayesian Inference Review: Hα in a Real SDSS Spectrum
Infer the redshift and $H\alpha$ flux of a single star-forming galaxy, then compare against SDSS's own pipeline measurements. 

We'll use spectra from **SDSS plate 2432, MJD 54052, fiber 327**, a star-forming galaxy at $z \approx 0.0205$ with strong $H\alpha$.

In [ ]:
import os
import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
import emcee
import corner
%matplotlib inline
np.random.seed(7)

## get the SDSS spectrum
We'll only keep the spectrum over the wavelength range $6300 < \lambda < 7100$

In [ ]:
PLATE, MJD, FIBER = 2432, 54052, 327
CACHE = f'data/sdss_spec_{PLATE}_{MJD}_{FIBER}.npz'

def load_spectrum():
    # use this function to download the spectra
    if os.path.exists(CACHE):
        return np.load(CACHE)
    # --- how the cache was built (needs network + astropy) ---
    import requests
    from astropy.io import fits
    url = (f'https://dr17.sdss.org/sas/dr17/sdss/spectro/redux/26/spectra/lite/'
           f'{PLATE:04d}/spec-{PLATE:04d}-{MJD}-{FIBER:04d}.fits')
    open('tmp.fits','wb').write(requests.get(url, timeout=90).content)
    hd = fits.open('tmp.fits')
    d = hd[1].data
    lam, flux, ivar = 10**d['loglam'], d['flux'], d['ivar']
    keep = (lam > 6300) & (lam < 7100) & (ivar > 0)
    os.makedirs('data', exist_ok=True)
    np.savez_compressed(CACHE, wavelength=lam[keep], flux=flux[keep], ivar=ivar[keep],
                        z_pipeline=float(hd[2].data['Z'][0]))
    return np.load(CACHE)

spec = load_spectrum()
lam  = spec['wavelength']          # Angstrom, VACUUM (see below)
flux = spec['flux']                # 1e-17 erg/s/cm^2/A
ivar = spec['ivar']                # inverse variance
sigma = 1/np.sqrt(ivar)            # per-pixel 1-sigma -- real data, varies pixel to pixel
z_pipeline = float(spec['z_pipeline'])

print(f'{len(lam)} pixels, {lam.min():.0f}-{lam.max():.0f} A')
print(f'SDSS pipeline redshift: z = {z_pipeline:.5f}')
print(f'per-pixel noise ranges {sigma.min():.2f} - {sigma.max():.2f} (median {np.median(sigma):.2f})')

In [ ]:
# vacuum rest wavelengths (Angstrom)
HALPHA  = 6564.61
NII_A   = 6549.86
NII_B   = 6585.27
SII_A   = 6718.29
SII_B   = 6732.68

ha_expected = HALPHA*(1 + z_pipeline)
print(f'Halpha expected at {ha_expected:.2f} A')

## Always look at the data first


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].plot(lam, flux, 'k-', lw=0.7)
axes[0].set_title('the region we downloaded')
for nm, lc, col, yfrac in [(r'H$\alpha$', HALPHA, 'C3', 1.02), ('[NII]', NII_A, 'C0', 0.80),
                           ('', NII_B, 'C0', 0.80), ('[SII]', SII_A, 'C2', 0.62),
                           ('', SII_B, 'C2', 0.62)]:
    axes[0].axvline(lc*(1+z_pipeline), color=col, ls='--', lw=1, alpha=0.8)
    if nm:
        axes[0].text(lc*(1+z_pipeline), flux.max()*yfrac, nm, color=col,
                     fontsize=9, ha='center', va='bottom')

w = np.abs(lam - ha_expected) < 70
axes[1].errorbar(lam[w], flux[w], yerr=sigma[w], fmt='.k', ms=4, lw=0.8)
axes[1].set_title(r'zoom on the H$\alpha$ complex')
for lc, col in [(HALPHA,'C3'), (NII_A,'C0'), (NII_B,'C0')]:
    axes[1].axvline(lc*(1+z_pipeline), color=col, ls='--', lw=1, alpha=0.8)

for ax in axes:
    ax.set_xlabel(r'observed wavelength [$\AA$]')
    ax.set_ylabel(r'flux [$10^{-17}$ erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$]')
plt.tight_layout(); plt.show()

Before we can fit the emission line to measure redshift and $H\alpha$, we need to remove the continuum.

We'll pick line-free "sidebands" around the complex, fit a low-order polynomial through them, and subtract. We mask a $\pm 12 A$ around every line so no emission leaks into the continuum estimate. For the purposes of this exercise you can ignore the details here. 

In [ ]:
CONT_HALFWIDTH = 60.0      # A, region considered for the continuum fit
LINE_MASK      = 12.0      # A, masked around each known line

near = np.abs(lam - ha_expected) < CONT_HALFWIDTH
line_mask = np.zeros_like(lam, dtype=bool)
for lc in [HALPHA, NII_A, NII_B, SII_A, SII_B]:
    line_mask |= np.abs(lam - lc*(1 + z_pipeline)) < LINE_MASK

cont_pix = near & ~line_mask
print(f'{cont_pix.sum()} continuum pixels out of {near.sum()} in the window')

# weighted first-order polynomial, centered so the coefficients are interpretable
pcoef = np.polyfit(lam[cont_pix] - ha_expected, flux[cont_pix], deg=1, w=1/sigma[cont_pix])
continuum = lambda l: np.polyval(pcoef, l - ha_expected)

print(f'continuum = {pcoef[1]:.2f} {pcoef[0]:+.4f} x (lambda - {ha_expected:.1f})')

resid = (flux[cont_pix] - continuum(lam[cont_pix]))/sigma[cont_pix]
print(f'standardized residuals in the sidebands: mean {resid.mean():+.2f}, std {resid.std():.2f}')

In [ ]:
flux_sub = flux - continuum(lam)      # continuum-subtracted spectrum

plt.figure(figsize=(9, 3.8))
plt.errorbar(lam[near], flux[near], yerr=sigma[near], fmt='.k', ms=4, lw=0.8, label='data')
plt.plot(lam[near], continuum(lam[near]), 'C1', lw=2, label='continuum fit')
plt.plot(lam[cont_pix], flux[cont_pix], 'o', ms=5, mfc='none', mec='C1',
         label='pixels used for continuum')
plt.xlabel(r'observed wavelength [$\AA$]'); plt.ylabel('flux')
plt.legend(frameon=False, fontsize=9); plt.show()

## The $H\alpha$ model

On the continuum-subtracted spectrum, a single Gaussian plus a small residual offset:

$$f(\lambda) = c_{\rm resid} + A\exp\!\left[-\frac{(\lambda-\lambda_c)^2}{2\sigma_\lambda^2}\right]$$

We'll focus solely on $H\alpha$. Rather than model all three lines, we'll fit a $\pm 40A$ window around $H\alpha$ and mask $\pm 9 A$ around each [N II] line. **You are welcome to fit all three lines jointly!**

In [ ]:
FIT_HALFWIDTH = 40.0
NII_MASK      = 9.0

in_window = np.abs(lam - ha_expected) < FIT_HALFWIDTH
nii_mask  = ((np.abs(lam - NII_A*(1+z_pipeline)) < NII_MASK) |
             (np.abs(lam - NII_B*(1+z_pipeline)) < NII_MASK))
use = in_window & ~nii_mask

L, Y, S = lam[use], flux_sub[use], sigma[use]
print(f'{use.sum()} pixels used, {(in_window & nii_mask).sum()} masked as [NII]')

def line_model(theta, l):
    ''' Here's our line model. The input parameter is 
    theta = (c_resid, amplitude, delta_lambda, sigma_lambda)
    '''
    c, A, dlam, s = theta
    return c + A*np.exp(-0.5*((l - (ha_expected + dlam))/s)**2)

## Set up your likelihood and posterior

In [ ]:
def log_prior(theta):
    # here's a default prior for you to use but feel free to change it
    c, A, dlam, s = theta
    if not (-50 < c < 50):      return -np.inf
    if not (0 < A < 2000):      return -np.inf
    if not (-25 < dlam < 25):   return -np.inf
    if not (0.3 < s < 15):      return -np.inf
    return 0.0

def log_likelihood(theta):
    # set up your Gaussian log-likelihood here
    return 

def log_posterior(theta):
    # set up your posterior 
    return

## initialize the walkers

## Sample the posterior using `emcee`
* Run EnsembleSampler
* check burn in
* check convergence
* plot the converged posterior using `corner`

In [ ]:
# run emcee EnsembleSampler here

## Derived quantities and compare with SDSS
Derive redshift, $H\alpha$ flux, and equivalent width. Calculate the 16, 50, 84th percentiles of their 1D marginal posterior. 

*The synatx below assumes that `flat` is the flattened MCMC chain*

In [ ]:
z_samp  = (ha_expected + flat[:, 2])/HALPHA - 1
F_samp  = flat[:, 1]*flat[:, 3]*np.sqrt(2*np.pi)          # 1e-17 erg/s/cm^2
EW_samp = F_samp/continuum(ha_expected + flat[:, 2])      # Angstrom

# derive 16, 50, 84th percentile


### How does our flux compare SDSS MPA-JHU catalog measurements
```
MPA_FLUX, MPA_FLUX_ERR = 1707.56, 11.47      # 1e-17 erg/s/cm^2
MPA_EQW = -35.28                              # A (negative = emission in SDSS convention)
```